# geo0441 – Geometries in Code: the Roman Camp, Bounding Boxes and Terrain Tiles

**Unit:** geo0440 GeoPandas: DTM tiles and bounding boxes · **Builds on:** geo0241 (georeferencing Castra Vetera, your polygon `castra_vetera.gpkg`), geo0221 (CRS in code) · **Data:** none to prepare; the NRW municipalities (DVG1) are read from your course data folder or from opengeodata.nrw.de

In QGIS you *digitized* the legionary fortress Vetera I by hand. Here you *construct* geometries from numbers, as a script does it: a model of the fortress, rectangles around it, and the 1 km terrain tiles of NRW that cover it. In the next notebook (geo0442) you download exactly these tiles.

You will learn to
1. build points, lines and polygons with **shapely** and read their area, length and bounds,
2. put geometries into a **GeoDataFrame** with the right CRS and look at them on a map,
3. compute **bounding boxes** and snap them to a 1 km grid,
4. find the **DTM tiles** that cover an area from their naming scheme,
5. **compare** two polygons with numbers (area, overlap, distance) and save layers for QGIS.

## 0. Setup

In [ ]:
import os
import re
from pathlib import Path
import numpy as np
import pandas as pd
import geopandas as gpd
import shapely
from shapely import affinity
from shapely.geometry import Point, LineString, Polygon, box

def course_data(start=None):
    """The data/ folder of your course folder (outside the session folders, ignored by git).
    OPENGEO_DATA if set; else the folder above the session folder sNN_date; else ./data."""
    env = os.environ.get("OPENGEO_DATA")
    if env:
        return Path(env).expanduser()
    start = Path.cwd() if start is None else Path(start)
    for folder in (start, *start.parents):
        if re.fullmatch(r"s\d{2}_\d{4}-\d{2}-\d{2}", folder.name):
            return folder.parent / "data"
    return Path("data")


CRS = "EPSG:25832"                  # ETRS89 / UTM zone 32N: metres, the CRS of NRW geodata
DATA = course_data()               # e.g. ~/iecs9544-ws2026/data (hub) or <course folder>/data (laptop)
OUT = DATA / "derived"              # results for QGIS (created below)
MY_POLYGON = Path("castra_vetera.gpkg")   # your geo0241 polygon (upload it next to this notebook)

# Vetera I as a simple model (rounded values, see §2)
CENTRE = (324_880, 5_724_300)       # E, N in metres (EPSG:25832)
LENGTH, WIDTH = 945, 625            # metres, long axis roughly north-south
ROTATION = 5                        # degrees, anticlockwise (long axis 5° west of grid north)
RADIUS = 30                         # rounded corners in metres (assumption)
TILE = 1000                         # NRW DTM tiles are 1 km x 1 km

OUT.mkdir(parents=True, exist_ok=True)
print("data folder:", DATA.resolve())
print("shapely", shapely.__version__, "| geopandas", gpd.__version__)

## 1. Points, lines, polygons
shapely knows nothing about maps: a geometry is a set of coordinates, and shapely computes as if the plane were flat. That is why we work in **metres (EPSG:25832)**, never in degrees (geo0221 §4).

**Predict:** a square with corners 100 m apart. What are its area and its perimeter?

In [ ]:
e, n = CENTRE
p = Point(e, n)                                          # the centre of the camp
line = LineString([(e, n - 300), (e, n + 300)])          # 600 m south -> north through the centre
square = Polygon([(e, n), (e + 100, n), (e + 100, n + 100), (e, n + 100)])   # corners, anticlockwise

print("point   ", p.wkt)
print("line    ", round(line.length), "m")
print("square  ", square.area, "m²,", square.length, "m perimeter")
print("bounds  ", square.bounds, " (minx, miny, maxx, maxy)")
square

Three things to remember:
* A polygon is a **closed ring**: shapely repeats the first corner at the end for you (`square.exterior.coords`).
* `.length` of a polygon is its **perimeter**, `.area` is in the square of the CRS unit (here m²).
* **WKT** (*well-known text*, `POINT (324880 5724300)`) is the same text format QGIS and PostGIS understand (*Copy feature* in QGIS, `ST_AsText` in PostGIS).

## 2. A model of the fortress
Roman legionary fortresses were laid out as a rectangle with rounded corners, the "playing-card" shape. We build Vetera I in three steps: rectangle → rounded corners → rotation.

### 2.1 Rectangle around the centre
`box(minx, miny, maxx, maxy)` builds an axis-parallel rectangle from its bounds.

In [ ]:
rect = box(e - WIDTH / 2, n - LENGTH / 2, e + WIDTH / 2, n + LENGTH / 2)
print(f"{rect.area / 10_000:.1f} ha")
rect

### 2.2 Rounded corners with a buffer
A **buffer** grows a geometry by a distance in every direction. Shrink the rectangle by the radius first, then buffer it by the radius: the sides end where they were, the corners become quarter circles.

**Predict:** is the area larger or smaller than the plain rectangle? By how much (formula)?

In [ ]:
inner = box(e - WIDTH / 2 + RADIUS, n - LENGTH / 2 + RADIUS, e + WIDTH / 2 - RADIUS, n + LENGTH / 2 - RADIUS)
card = inner.buffer(RADIUS)            # quarter circles at the corners (approximated by short segments)

print(f"rectangle {rect.area / 10_000:.2f} ha, playing card {card.area / 10_000:.2f} ha")
print(f"difference {rect.area - card.area:.0f} m², formula (4 - pi) r² = {(4 - np.pi) * RADIUS**2:.0f} m²")
card

### 2.3 Rotation and the GeoDataFrame
(The small gap between the two numbers above: the buffer approximates the quarter circles by short straight segments.)

The long axis of the camp is not exactly north-south. `affinity.rotate` turns a geometry around a point; positive angles are **anticlockwise**.

A shapely geometry has no CRS. Only the **GeoDataFrame** carries one, and we *assign* it (geo0221 §2): the numbers are already EPSG:25832.

In [ ]:
camp = affinity.rotate(card, ROTATION, origin=CENTRE)

model = gpd.GeoDataFrame(
    {"name": ["Vetera I (model)"],
     "source": [f"{LENGTH} m x {WIDTH} m, rotated {ROTATION} deg, corner radius {RADIUS} m (rounded values)"]},
    geometry=[camp], crs=CRS)                       # assign: the numbers are metres in UTM 32N
model["area_ha"] = (model.area / 10_000).round(1)
model

Check it on a map with the DTM hillshade of Geobasis NRW (the same WMS as in geo0921 Part A). `explore()` reprojects to Web Mercator for the web map by itself.

**Look:** does the model sit on the Fürstenberg plateau? Where are the edges of the terrace?

In [ ]:
import folium

m = model.explore(color="red", style_kwds={"fillOpacity": 0.1, "weight": 3}, zoom_start=15)
folium.WmsTileLayer(url="https://www.wms.nrw.de/geobasis/wms_nw_dgm-schummerung",
                    layers="nw_dgm-schummerung_pan", fmt="image/png", transparent=True,
                    name="DTM hillshade NRW", attr="© Geobasis NRW, dl-de/zero-2-0").add_to(m)
folium.LayerControl().add_to(m)
m

## 3. Bounding boxes
A **bounding box** (BB) is the smallest axis-parallel rectangle that contains a geometry: four numbers `(minx, miny, maxx, maxy)`. Downloads, map extents, tile searches and spatial indexes all work with bounding boxes.

### 3.1 Three rectangles around the camp
**Predict:** the camp is 945 m × 625 m. Is its bounding box larger, equal or smaller?

In [ ]:
boxes = gpd.GeoDataFrame(
    {"kind": ["bounding box (envelope)", "minimum rotated rectangle"]},
    geometry=[camp.envelope, camp.minimum_rotated_rectangle], crs=CRS)
boxes["width_m"] = boxes.bounds.eval("maxx - minx").round()
boxes["height_m"] = boxes.bounds.eval("maxy - miny").round()
boxes["area_ha"] = (boxes.area / 10_000).round(1)
print("camp bounds:", np.round(camp.bounds))
boxes

The envelope is larger than the camp because the camp is rotated; the minimum rotated rectangle follows the camp's own axes (its width and height columns show the envelope of that rectangle, not its side lengths).

### 3.2 Snap a bounding box to the 1 km grid
For the DTM tiles we need a box whose corners are **multiples of 1000 m** and which still contains the whole area: round the lower left corner *down*, the upper right corner *up*.

**Exercise:** write `snap_bounds(bounds, step=TILE)`: it takes `(minx, miny, maxx, maxy)` and returns the same box with the lower left corner rounded *down* and the upper right corner rounded *up* to multiples of `step`. Test: `snap_bounds((317215.4, 5721035.1, 328192.1, 5737002.2))` must give `(317000, 5721000, 329000, 5738000)`. What happens to a value that already is a multiple of 1000?

In [ ]:
def snap_bounds(bounds, step=TILE):
    """(minx, miny, maxx, maxy) -> the same box with corners rounded outwards to multiples of `step`."""
    # YOUR CODE HERE


assert snap_bounds((317215.4, 5721035.1, 328192.1, 5737002.2)) == (317000, 5721000, 329000, 5738000)
snap_bounds(camp.bounds)

### 3.3 The municipality of Xanten
Now a real polygon: the municipality boundary from DVG1, the official administrative boundaries of NRW. `read_dvg1` takes your local copy if it finds one, otherwise it reads the layer straight out of the zip file on opengeodata.nrw.de (GDAL only fetches the bytes it needs; about 20 s).

In [ ]:
DVG1_ZIP = "https://www.opengeodata.nrw.de/produkte/geobasis/vkg/dvg/dvg1/dvg1_EPSG25832_Shape.zip"
DVG1_LOCAL = [DATA / "dvg1_EPSG25832_Shape", Path.home() / "shared/data/opengeo/dvg1"]


def read_dvg1(layer="dvg1gem_nw", where=None):
    """A DVG1 layer (EPSG:25832) from a local copy or directly from the zip on opengeodata.nrw.de."""
    for folder in DVG1_LOCAL:
        if (folder / f"{layer}.shp").exists():
            print("local:", folder / f"{layer}.shp")
            return gpd.read_file(folder / f"{layer}.shp", where=where)
    print("reading from", DVG1_ZIP)
    return gpd.read_file(f"/vsizip//vsicurl/{DVG1_ZIP}", layer=layer, where=where)


xanten = read_dvg1("dvg1gem_nw", where="GN = 'Xanten'")
xanten[["GN", "KN"]].assign(area_km2=(xanten.area / 1e6).round(1), crs=str(xanten.crs))

**Exercise (this was the 2022 task):** build a GeoDataFrame `xanten_bb` with **one** polygon: the bounding box of Xanten snapped to the 1 km grid. Give it a text column `label` in the form `E:317000,N:5721000` (the lower left corner) and a column `area_km2`. How many km² larger than the municipality is the box?

In [ ]:
# YOUR CODE HERE
print(f"box {xanten_bb.area_km2.iloc[0]:.0f} km², municipality {xanten.area.sum() / 1e6:.1f} km²")
xanten_bb

## 4. Which terrain tiles cover the camp?
Geobasis NRW publishes the digital terrain model (DGM1, 1 m resolution) as **1 km × 1 km GeoTIFF tiles**. The file name *is* the bounding box:

`dgm1_32_324_5724_1_nw_2025.tif` = DGM 1 m, UTM zone **32**, lower left corner **E 324 km, N 5724 km**, tile size **1** km, **nw** = NRW, survey year 2025.

### 4.1 From a tile name to its polygon

In [ ]:
def tile_box(e_km, n_km, size_km=1):
    """Polygon of the DTM tile with the lower left corner (e_km, n_km) in kilometres, EPSG:25832."""
    return box(e_km * 1000, n_km * 1000, (e_km + size_km) * 1000, (n_km + size_km) * 1000)


def tile_key(e_km, n_km):
    """Tile name without the survey year (the year differs from tile to tile)."""
    return f"dgm1_32_{e_km}_{n_km}_1_nw"


name = "dgm1_32_324_5724_1_nw_2025.tif"
parts = name.split("_")                 # ['dgm1', '32', '324', '5724', '1', 'nw', '2025.tif']
e_km, n_km = int(parts[2]), int(parts[3])
print(tile_key(e_km, n_km), tile_box(e_km, n_km).bounds)

### 4.2 All tiles that touch an area
Snap the bounding box to the grid, step through it in 1 km steps, and keep the tiles that **intersect** the area (a bounding box can contain tiles that the area itself does not touch).

**Predict:** how many tiles does the camp touch? Look at the numbers in `camp.bounds` above.

**Exercise:** write `tiles_covering(geom)`. It returns a GeoDataFrame with the columns `key` (`tile_key`) and `geometry` (`tile_box`) of all 1 km tiles that **intersect** `geom`. Use `snap_bounds` and two loops over the kilometre numbers.

In [ ]:
def tiles_covering(geom, crs=CRS):
    """GeoDataFrame of the 1 km DTM tiles (key, polygon) that intersect `geom` (EPSG:25832)."""
    # YOUR CODE HERE


tiles_camp = tiles_covering(camp)
tiles_camp

### 4.3 The surroundings and the whole municipality
For the terrain around the camp (the edge of the Fürstenberg, the old Rhine channel) we want a margin of 500 m: buffer first, then search. Then the same for Xanten.

**Predict:** Xanten is about 11 km × 16 km. How many tiles are in its snapped bounding box, and how many does the municipality really touch?

In [ ]:
tiles_area = tiles_covering(camp.buffer(500))
tiles_xanten = tiles_covering(xanten.union_all())
bb_tiles = int((xanten_bb.area_km2.iloc[0]))
print(f"camp: {len(tiles_camp)} tiles, camp + 500 m: {len(tiles_area)} tiles")
print(f"Xanten: {len(tiles_xanten)} of {bb_tiles} tiles in the bounding box "
      f"(about {len(tiles_xanten) * 2.4 / 1000:.1f} GB of GeoTIFF at ~2.4 MB per tile)")

### 4.4 Look ahead: which survey year? (geo0442)
The year at the end of the file name is only known from the server's file list, an XML page with about 36 000 entries. In geo0442 you read it with Python (*scraping*) and download the tiles. Here a first look (optional, about 4 MB):

In [ ]:
import re
import requests

DGM1_INDEX = "https://www.opengeodata.nrw.de/produkte/geobasis/hm/dgm1_tiff/dgm1_tiff/"
names = re.findall(r'file name="(dgm1_[^"]+\.tif)"', requests.get(DGM1_INDEX, timeout=60).text)
by_key = {nm.rsplit("_", 1)[0]: nm for nm in names}        # key without the year -> full file name
print(len(names), "tiles in NRW")
tiles_camp.assign(file=tiles_camp["key"].map(by_key))

## 5. Model, map and your digitization
In geo0241 you digitized the area that the 1998 map marks as cultural monument (*KD*). How well do model and digitization agree? Upload your `castra_vetera.gpkg` from the laptop into the folder of this notebook (JupyterLab: *Upload* button) and run the cells.

Measures for two polygons A and B:
* **area** of each, and their ratio;
* **IoU** (*intersection over union*) = area(A ∩ B) / area(A ∪ B): 1 = identical, 0 = no overlap;
* **Hausdorff distance**: the largest distance from a point of one outline to the nearest point of the other ("the worst spot", in metres).

**Predict:** which IoU do you expect for a careful digitization? Where will the two outlines differ most?

In [ ]:
if MY_POLYGON.exists():
    mine = gpd.read_file(MY_POLYGON).to_crs(CRS)          # transform, in case you saved it in another CRS
    my_camp = mine.union_all()                             # one geometry, even if you digitized several parts
    print(f"{len(mine)} feature(s), CRS {mine.crs.to_epsg()}, {my_camp.area / 10_000:.1f} ha")
else:
    my_camp = None
    print(f"{MY_POLYGON} not found: upload your geo0241 GeoPackage next to this notebook (or set MY_POLYGON)")

**Exercise:** write `compare(a, b)`. It returns a dictionary with the areas of both polygons in ha, the IoU, the Hausdorff distance and the distance between the centroids in metres (shapely: `intersection`, `union`, `hausdorff_distance`, `centroid`).

In [ ]:
def compare(a, b):
    """Area of a and b (ha), IoU and Hausdorff distance (m) of two polygons in a metric CRS."""
    # YOUR CODE HERE


if my_camp is not None:
    print(compare(camp, my_camp))

**Interpret** (one sentence each, in your report):
1. Which is "right", the model or your polygon? (Neither: both are models. What does each one represent?)
2. Where is the Hausdorff distance reached? Show it in QGIS (§6) and give a reason: digitizing, georeferencing (your RMS), the map itself, or the model's assumptions?
3. Change `RADIUS` or `ROTATION` in §0 and run §2 and §5 again: which value improves the IoU most?

✍️ *Your answer:*

## 6. Hand over to QGIS
One GeoPackage can hold many layers. Save the model, the boxes and the tiles in `data/derived/geo0441_vetera.gpkg` (the `data/` folder of your course folder, printed in §0) and open it in QGIS together with your georeferenced map and the hillshade WMS.

In [ ]:
gpkg = OUT / "geo0441_vetera.gpkg"
model.to_file(gpkg, layer="camp_model", driver="GPKG")
boxes.to_file(gpkg, layer="camp_boxes", driver="GPKG")
tiles_area.to_file(gpkg, layer="dtm_tiles_camp_500m", driver="GPKG")
xanten_bb.to_file(gpkg, layer="xanten_bb", driver="GPKG")
tiles_xanten.to_file(gpkg, layer="dtm_tiles_xanten", driver="GPKG")

import pyogrio
pyogrio.list_layers(gpkg)

**GeoJSON** is the format of web maps (geojson.io, Leaflet). Its standard (RFC 7946) only allows **WGS 84 longitude/latitude**: transform before you write it.

**Predict:** how many decimal places of a degree do you need for 1 m on the ground?

In [ ]:
geojson = OUT / "geo0441_camp_model.geojson"
model.to_crs("EPSG:4326").to_file(geojson, driver="GeoJSON", COORDINATE_PRECISION=6)
print(geojson.read_text(encoding="utf-8")[:400], "...")

In QGIS (laptop): *Layer → Add Layer → Add Vector Layer…* → `geo0441_vetera.gpkg` → select all layers. If you work on the hub, download the file first (right-click → *Download*) and save it in your `s04` folder.

**Check in QGIS:**
* The tiles lie exactly on the 1 km grid lines of the TK25 photo you georeferenced (the blue UTM grid).
* `camp_model` and your polygon: does the difference match the Hausdorff distance from §5?

## 7. Wrap-up
1. A geometry is just coordinates; the **CRS** belongs to the GeoDataFrame. Compute areas, lengths and buffers in a metric CRS.
2. `box`, `buffer`, `rotate`, `envelope` and `intersects` are enough to construct and select most rectangles you meet in GIS work.
3. A **bounding box** is four numbers; snapped to a grid it tells you which **tiles** to download.
4. Two polygons are compared with numbers: area, **IoU**, Hausdorff distance, not by eye only.
5. GeoPackage keeps many layers and any CRS; GeoJSON wants WGS 84.

**Exit ticket:** the bounding box of a rotated rectangle is larger than the rectangle. When is it exactly as large?

**Homework:** see the course plan in the README of your course repository.